# Define Constants Here

In [1]:
# if using google colab, upload the zip file and then run this cell to unzip the file.
!unzip PianoFingeringDataset_v1.2.zip

Archive:  PianoFingeringDataset_v1.2.zip
   creating: PianoFingeringDataset_v1.2/
  inflating: __MACOSX/._PianoFingeringDataset_v1.2  
  inflating: PianoFingeringDataset_v1.2/.DS_Store  
  inflating: __MACOSX/PianoFingeringDataset_v1.2/._.DS_Store  
   creating: PianoFingeringDataset_v1.2/FingeringFiles/
  inflating: __MACOSX/PianoFingeringDataset_v1.2/._FingeringFiles  
  inflating: PianoFingeringDataset_v1.2/List.pdf  
  inflating: __MACOSX/PianoFingeringDataset_v1.2/._List.pdf  
  inflating: PianoFingeringDataset_v1.2/List.csv  
  inflating: __MACOSX/PianoFingeringDataset_v1.2/._List.csv  
  inflating: PianoFingeringDataset_v1.2/README-ja.pdf  
  inflating: __MACOSX/PianoFingeringDataset_v1.2/._README-ja.pdf  
  inflating: PianoFingeringDataset_v1.2/README.pdf  
  inflating: __MACOSX/PianoFingeringDataset_v1.2/._README.pdf  
   creating: PianoFingeringDataset_v1.2/ScorePDF/
  inflating: __MACOSX/PianoFingeringDataset_v1.2/._ScorePDF  
  inflating: PianoFingeringDataset_v1.2/Fingerin

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from collections import defaultdict

import proj_consts_
# from proj_consts_ import *

# easy conversion between pitch and int token, and finger and int token
# print("pitch_to_int_mapping: ", finger_mappings.pitch_to_int_mapping)
# print("int_to_pitch_mapping: ", finger_mappings.int_to_pitch_mapping)

# print("finger_to_int_mapping: ", finger_mappings.finger_to_int_mapping)
# print("int_to_finger_mapping: ", finger_mappings.int_to_finger_mapping)

pitch_to_int_mapping = proj_consts_.pitch_to_int_mapping
int_to_pitch_mapping = proj_consts_.int_to_pitch_mapping
finger_to_int_mapping = proj_consts_.finger_to_int_mapping
int_to_finger_mapping = proj_consts_.int_to_finger_mapping

In [4]:
import os
import shutil
import random

# Define dataset folder path
PATH_TO_DATASET_FOLDER = './PianoFingeringDataset_v1.2/FingeringFiles'

# Define output folders
OUTPUT_FOLDER = "./fine_tuning_dataset"
TRAIN_FOLDER = os.path.join(OUTPUT_FOLDER, "train")
TEST_FOLDER = os.path.join(OUTPUT_FOLDER, "test")

# Ensure output directories exist
os.makedirs(TRAIN_FOLDER, exist_ok=True)
os.makedirs(TEST_FOLDER, exist_ok=True)

# Get all files in the dataset folder
all_files = [f for f in os.listdir(PATH_TO_DATASET_FOLDER) if f.endswith("_fingering.txt")]

# Shuffle for randomness
random.seed(42)  # Set seed for reproducibility
random.shuffle(all_files)

# Split into train (80%) and test (20%)
split_idx = int(len(all_files) * 0.8)
train_files = all_files[:split_idx]
test_files = all_files[split_idx:]

# Move files into train and test folders
for file in train_files:
    shutil.copy(os.path.join(PATH_TO_DATASET_FOLDER, file), os.path.join(TRAIN_FOLDER, file))

for file in test_files:
    shutil.copy(os.path.join(PATH_TO_DATASET_FOLDER, file), os.path.join(TEST_FOLDER, file))

# Summary
print(f"✅ Split completed! {len(train_files)} train files, {len(test_files)} test files.")
print(f"📂 Train files are in: {TRAIN_FOLDER}")
print(f"📂 Test files are in: {TEST_FOLDER}")


✅ Split completed! 247 train files, 62 test files.
📂 Train files are in: ./fine_tuning_dataset/train
📂 Test files are in: ./fine_tuning_dataset/test


In [5]:
# PATH_TO_DATASET_FOLDER = './ThumbSet_v1.0/'   # PRETRAINING DATASET
# PATH_TO_DATASET_FOLDER = './Split_Fingering_Dataset/train/HI/'   # FINE-TUNING DATASET ON A SPECIFIC ANNOTATOR
# PATH_TO_METADATA = './PianoFingeringDataset_v1.2/List.csv'
# EVAL_DATASET_PATH = './Split_Fingering_Dataset/test/HI/'

PATH_TO_DATASET_FOLDER = './fine_tuning_dataset/train'   # FINE-TUNING DATASET ON A SPECIFIC ANNOTATOR
PATH_TO_METADATA = './PianoFingeringDataset_v1.2/List.csv'
EVAL_DATASET_PATH = './fine_tuning_dataset/test'

# Define Dataset Here

We are treating this as a **translation task** where we want to map sequences-to-sequences, rather than treating it like an autoregressive model. For that reason, we will be using the __entire pieces__ as the sequence training data.

In [6]:
directory_path = PATH_TO_DATASET_FOLDER

all_sequences = []   # will hold raw DataFrames for each piece
all_piece_names = []


# you can set 
# FINGERING_TYPE_TO_ANALYZE = "HI"
# yi_filenames = set(proj_consts_.annotator_to_files_dict.get(FINGERING_TYPE_TO_ANALYZE, []))

# print(yi_filenames)

print(directory_path)


for filename in os.listdir(directory_path):
    file_path = os.path.join(directory_path, filename)
    fingering_label, _ = filename.split('_')  # e.g. "001-01"
    piece_id, fingering_type = fingering_label.split('-')

    # if os.path.isfile(file_path) and (fingering_label in yi_filenames):
    if os.path.isfile(file_path):
        # print(fingering_label)
        df = pd.read_table(
            file_path, sep="\t", skiprows=1,
            names=[
                "noteID", "onset_time", "offset_time", "spelled_pitch",
                "onset_velocity", "offset_velocity", "channel",
                "finger_number"
            ]
        )
        all_piece_names.append(filename)
        all_sequences.append(df)

print(f"Total number of pieces found: {len(all_sequences)}")
raw_encoded_sequences_variant0 = []  # STORES TUPLES OF (X_SEQ, Y_LIST_OF_FINGERINGS_FOR_EVERY_X_IN_THE_SEQ)

for df in all_sequences:
    X_list = []
    y_list = []
    for row in df.itertuples(index=False):
        spelled_pitch = row.spelled_pitch
        finger_str = str(row.finger_number)

        # encode pitch and finger
        pitch_int = pitch_to_int_mapping.get(spelled_pitch, 0)  # fallback 0 if unseen
        finger_int = finger_to_int_mapping.get(finger_str, 0)   # fallback 0 if unseen
        feature_row = [
            pitch_int,
            float(row.onset_time),
            float(row.offset_time),
            float(row.onset_velocity),
            float(row.offset_velocity),
            float(row.channel)
        ]
        X_list.append(feature_row)
        y_list.append(finger_int)

    raw_encoded_sequences_variant0.append((X_list, y_list))


# raw_encoded_sequences_variant0

./fine_tuning_dataset/train
Total number of pieces found: 247


In [10]:
raw_encoded_sequences = raw_encoded_sequences_variant0
print("x sequence length: ", len(raw_encoded_sequences[0][0]))
print("y sequence length (should match): ", len(raw_encoded_sequences[0][-1]))

# potentially do scaling/some kind of layernorm???
train_val_seqs, test_seqs = train_test_split(raw_encoded_sequences, test_size=0.05, random_state=42)
train_seqs, val_seqs = train_test_split(train_val_seqs, test_size=0.05, random_state=42)

print(f"Total pieces: {len(raw_encoded_sequences)}")
print(f"Train pieces: {len(train_seqs)}")
print(f"Val pieces:   {len(val_seqs)}")
print(f"Test pieces:  {len(test_seqs)}")

x sequence length:  289
y sequence length (should match):  289
Total pieces: 247
Train pieces: 222
Val pieces:   12
Test pieces:  13


In [11]:
# data loader component
# DATASET LOADER CLASS

class FingeringDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        X_list, y_list = self.sequences[idx]
        X_tensor = torch.tensor(X_list, dtype=torch.float32)  # (seq_len, 6)
        y_tensor = torch.tensor(y_list, dtype=torch.long)     # (seq_len,)
        return X_tensor, y_tensor

train_dataset = FingeringDataset(train_seqs)
val_dataset   = FingeringDataset(val_seqs)
test_dataset  = FingeringDataset(test_seqs)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=1, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=1, shuffle=False)

# Modify Architecture and Main Training Loop

In [12]:
# training loop
def train_sequence_model(model, train_loader, val_loader, num_epochs=5, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        for X_seq, y_seq in train_loader:
            # print("X_seq: ", X_seq)
            # print("y_seq: ", y_seq)

            X_seq = X_seq.to(device, non_blocking=True)
            y_seq = y_seq.to(device, non_blocking=True)

            optimizer.zero_grad()
            logits = model(X_seq)    # forward pass

            # reshape for CrossEntropy by flattening tokens => (batch_size * seq_len, num_fingers)
            logits_reshaped = logits.view(-1, logits.size(-1))
            y_seq_reshaped = y_seq.view(-1)

            loss = criterion(logits_reshaped, y_seq_reshaped)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        val_acc = evaluate_sequence_model(model, val_loader)
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f} | ValAcc: {val_acc:.4f}")

# doing inference for validation
def evaluate_sequence_model(model, data_loader):
    device = next(model.parameters()).device
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_seq, y_seq in data_loader:
            # X_seq: (batch_size, seq_len, num_features)
            # y_seq: (batch_size, seq_len)

            X_seq = X_seq.to(device, non_blocking=True)
            y_seq = y_seq.to(device, non_blocking=True)

            logits = model(X_seq)  # => (batch_size, seq_len, num_fingers)
            preds = torch.argmax(logits, dim=-1)  # => (batch_size, seq_len)

            all_preds.extend(preds.view(-1).cpu().numpy())
            all_labels.extend(y_seq.view(-1).cpu().numpy())

    return accuracy_score(all_labels, all_preds)


In [13]:
# RNN MODEL
class PianoRNNModel(nn.Module):
    def __init__(
        self,
        num_pitches,     # size of pitch vocab
        num_fingers,     # number of finger classes
        embed_dim=32,    # dimension for pitch embedding
        hidden_dim=64,   # dimension for RNN hidden layer
        numeric_dim=5,   # number of extra numeric features
    ):
        super().__init__()

        # have 2 embedding vectors: one for pitch and the other numeric features. We then concatenate them and that ends up being the hidden state of our RNN.
        self.embedding_pitch = nn.Embedding(num_pitches, embed_dim)        # Embedding for pitch (categorical)
        self.embedding_numeric = nn.Linear(numeric_dim, embed_dim)     # A simple linear projection for numeric features => same embed_dim so we can concatenate them with pitch embeddings

        self.rnn = nn.RNN(embed_dim * 2, hidden_dim, batch_first=True)         # RNN: input_dim = embed_dim(pitch) + embed_dim(numeric) = 2*embed_dim
        self.fc = nn.Linear(hidden_dim, num_fingers)   # Final linear layer to map hidden_dim => number of finger classes

    def forward(self, x):
        pitch_ids = x[..., 0].long()         # shape => (batch_size, seq_len)
        numeric_feats = x[..., 1:].float()   # shape => (batch_size, seq_len, 5)

        # Embed pitch and numeric separately
        pitch_emb = self.embedding_pitch(pitch_ids)  # (batch_size, seq_len, embed_dim)
        numeric_emb = self.embedding_numeric(numeric_feats)  # (batch_size, seq_len, embed_dim)

        # Concatenate along last dimension => (batch_size, seq_len, 2*embed_dim)
        combined_emb = torch.cat((pitch_emb, numeric_emb), dim=2)

        # RNN => (batch_size, seq_len, hidden_dim)
        rnn_out, _ = self.rnn(combined_emb)

        # Final linear => (batch_size, seq_len, num_fingers)
        logits = self.fc(rnn_out)
        return logits

In [26]:
# COMMENT OUT THESE LINES WHEN YOU WANT TO CONTINUE TRAINING WITHOUT OVERWRITING YOUR PREVIOUS GAINS!
rnn_model = PianoRNNModel(
    num_pitches=len(pitch_to_int_mapping),
    num_fingers=len(finger_to_int_mapping),
    embed_dim=32,
    hidden_dim=64,
    numeric_dim=5
)

state_dict = torch.load('rnn_model_weights_FINETUNED.pth')
rnn_model.load_state_dict(state_dict)

train_sequence_model(rnn_model, train_loader, val_loader, num_epochs=50, lr=1e-3)
test_acc_rnn = evaluate_sequence_model(rnn_model, test_loader)
print(f"RNN Test Accuracy = {test_acc_rnn:.4f}")
torch.save(rnn_model.state_dict(), "rnn_model_weights_FINETUNED.pth")    # we save the RNN model weights at the very end so we can load them in later!

/tmp/ipykernel_25842/978097882.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('rnn_model_weights_FINETUNED.pth')


Epoch 1/50 | Loss: 0.9757 | ValAcc: 0.6115
Epoch 2/50 | Loss: 0.9729 | ValAcc: 0.6163
Epoch 3/50 | Loss: 0.9687 | ValAcc: 0.6143
Epoch 4/50 | Loss: 0.9695 | ValAcc: 0.6023
Epoch 5/50 | Loss: 0.9674 | ValAcc: 0.6080
Epoch 6/50 | Loss: 0.9675 | ValAcc: 0.6103
Epoch 7/50 | Loss: 0.9715 | ValAcc: 0.6053
Epoch 8/50 | Loss: 0.9614 | ValAcc: 0.6008
Epoch 9/50 | Loss: 0.9705 | ValAcc: 0.6003
Epoch 10/50 | Loss: 0.9557 | ValAcc: 0.6050
Epoch 11/50 | Loss: 0.9571 | ValAcc: 0.6098
Epoch 12/50 | Loss: 0.9623 | ValAcc: 0.6063
Epoch 13/50 | Loss: 0.9514 | ValAcc: 0.6150
Epoch 14/50 | Loss: 0.9561 | ValAcc: 0.5966
Epoch 15/50 | Loss: 0.9556 | ValAcc: 0.6073
Epoch 16/50 | Loss: 0.9529 | ValAcc: 0.6078
Epoch 17/50 | Loss: 0.9618 | ValAcc: 0.6262
Epoch 18/50 | Loss: 0.9506 | ValAcc: 0.6038
Epoch 19/50 | Loss: 0.9414 | ValAcc: 0.6093
Epoch 20/50 | Loss: 0.9519 | ValAcc: 0.6135
Epoch 21/50 | Loss: 0.9445 | ValAcc: 0.6100
Epoch 22/50 | Loss: 0.9423 | ValAcc: 0.6190
Epoch 23/50 | Loss: 0.9391 | ValAcc: 0.60

# Run Evaluation Metrics

In [27]:
import proj_consts_
from proj_consts_ import FingeringEvaluator

# Load the trained model weights and set up the model
rnn_model = PianoRNNModel(
    num_pitches=len(pitch_to_int_mapping),
    num_fingers=len(finger_to_int_mapping),
    embed_dim=32,
    hidden_dim=64,
    numeric_dim=5
)
state_dict = torch.load('rnn_model_weights_FINETUNED.pth')
rnn_model.load_state_dict(state_dict)
rnn_model.eval()  # set to evaluation mode

# Load test data via the evaluator
evaluator = FingeringEvaluator(EVAL_DATASET_PATH, PATH_TO_DATASET_FOLDER, PATH_TO_METADATA)

ground_truth_fingerings, piece_ids, lengths = evaluator.load_test_data()
# print(ground_truth_fingerings)
# print(piece_ids)

all_predicted_fingerings = []
all_ground_truth_fingerings = []
all_ids = []
all_lengths = []

# Iterate over each piece/annotator combination
for (piece_id, annotator_id), gt_fingering, length in zip(piece_ids, ground_truth_fingerings, lengths):
    # Construct the file name based on your naming convention
    file_name = f"{piece_id}-{annotator_id}_fingering.txt"
    file_path = os.path.join(EVAL_DATASET_PATH, file_name)
    
    if not os.path.exists(file_path):
        print(f"File {file_path} does not exist. Skipping.")
        continue

    # Use your multi-piece prediction helper function on this piece.
    rnn_preds, rnn_true, rnn_pred_labels, rnn_true_labels, rnn_acc = proj_consts_.predict_piece_fingerings_multi(
        model=rnn_model,
        file_path=file_path,
        pitch_to_idx=pitch_to_int_mapping,
        finger_to_idx=finger_to_int_mapping,
        idx_to_finger=int_to_finger_mapping
    )
    
    # Collect predictions and corresponding ground truth for later evaluation.
    all_predicted_fingerings.append(rnn_preds)
    all_ground_truth_fingerings.append(gt_fingering)
    all_ids.append((piece_id, annotator_id))
    all_lengths.append(length)

print(f"all_predicted_fingerings: {all_predicted_fingerings[0]}")
print(f"all_ground_truth_fingerings: {all_ground_truth_fingerings[0]}")
print(f"all_ids: {all_ids[0]}")
print(f"all_lengths: {all_lengths[0]}")

# Evaluate the model over all pieces
results = proj_consts_.evaluate_fingering_method(
    predicted_fingerings=all_predicted_fingerings, 
    ground_truth_fingerings=all_ground_truth_fingerings, 
    piece_ids=all_ids, 
    lengths=all_lengths, 
    method_name="RNN Model",
    evaluator=evaluator
)

evaluator.print_results(results, method_name="RNN Model")

/tmp/ipykernel_25842/3593136743.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('rnn_model_weights_FINETUNED.pth')


./fine_tuning_dataset/test
[[0, 14, 10, 12, 10, 2, 14, 10, 4, 12, 10, 0, 13, 10, 11, 1, 10, 2, 0, 14, 10, 12, 10, 2, 14, 10, 11, 10, 1, 14, 10, 4, 12, 10, 2, 14, 10, 1, 12, 2, 1, 0, 10, 2, 1, 13, 10, 11, 10, 0, 11, 1, 2, 0, 1, 2, 3, 4, 3, 2, 1, 0, 13, 10, 3, 2, 1, 0, 14, 10, 1, 2, 0, 1, 2, 3, 4, 3, 2, 1, 0, 13, 10, 3, 2, 1, 0, 14, 10, 1, 2, 0, 1, 2, 3, 4, 3, 2, 1, 0, 13, 10, 3, 2, 1, 0, 14, 10, 1, 2, 0, 1, 2, 3, 4, 3, 2, 1, 0, 14, 12, 3, 2, 1, 0, 11, 10, 1, 2, 0, 1, 2, 3, 4, 0, 1, 2, 0, 1, 2, 0, 1, 12, 2, 3, 2, 1, 0, 2, 11, 1, 2, 10, 3, 4, 3, 2, 1, 0, 11, 1, 0, 14, 12, 4, 11, 10, 2, 14, 12, 0, 11, 10, 1, 14, 12, 4, 11, 10, 2, 14, 12, 0, 11, 10, 1, 14, 0, 1, 4, 10, 0, 14], [4, 10, 12, 0, 1, 0, 42, 11, 90, 0, 1, 0, 11, 0, 1, 0, 11, 0, 1, 0, 3, 11, 14, 0, 4, 13, 0, 50, 11, 14, 0, 1, 0, 4, 11, 13, 0, 4, 12, 0, 4, 10, 11, 0, 1, 0, 2, 10, 13, 0, 1, 0, 4, 10, 11, 0, 1, 3, 10, 12, 0, 1, 0, 4, 11, 14, 0, 1, 0, 11, 0, 1, 0, 3, 11, 0, 1, 0, 4, 11, 14, 0, 3, 13, 0, 4, 10, 12, 0, 1, 0, 3, 11, 14, 0